# Axis III, RQ3 -- 50 subjects (TotalSegmentatorV2)

Implements **only RQ3** on the TotalSegmentatorV2 50-subject dataset. This notebook runs
**the first 50 loaded subjects** in one pass -- no batching.

**RQ3 -- Does the prompt decoder preserve or suppress laterality information?**

Uses the **TotalSegmentatorV2 50-subject dataset** (`Fatema142/totalsegmentator-50-rq-analysis`
on Hugging Face).

The VoxTell setup (environment install, model loading, Qwen3 text-encoder fix) pins whatever
numpy/scipy/torch/transformers/etc. Kaggle's kernel already has loaded via a pip constraints
file, installs voxtell with `--no-deps`, and only adds the runtime deps actually missing
(nnunetv2, acvl-utils, etc.). Nothing already-loaded-in-memory gets replaced on disk, so no
kernel restart is needed and "Save & Run All" works end to end.

| RQ | Question (short) | Folder | GPU cost |
|---|---|---|---|
| RQ3 | Does the prompt decoder preserve or suppress laterality? | `results/rq3/` | 1 forward pass/side/organ |


## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Setup working directories (Kaggle environment)

In [ ]:
import os
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
for _rq in ('rq1', 'rq2', 'rq3', 'rq4', 'rq5', 'rq6', 'rq7', 'rq8', 'rq9', 'rq10', 'rq11'):
    (RESULTS_ROOT / _rq).mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT, 'in subfolders rq1/ ... rq11/')


## 3. Install VoxTell

In [ ]:
import subprocess, sys, importlib
from importlib import metadata
from pathlib import Path
import os

# Reduce CUDA memory fragmentation (must be set before the first CUDA allocation)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Installs VoxTell + its missing dependencies WITHOUT touching any compiled package that
# Kaggle's kernel already has loaded in memory (numpy, scipy, torch, ...). Replacing those
# on disk mid-session is what caused errors like:
#   ImportError: cannot import name '_slice' from 'numpy._core.umath'
# because a running process cannot swap a loaded C-extension. By pinning them to the
# versions already installed, no kernel restart is needed and "Save Version" works.
#
# voxtell>=0.1.2 accepts transformers<6, so Kaggle's preinstalled transformers is kept too.

_PIN_IF_PRESENT = [
    'numpy', 'scipy', 'torch', 'torchvision', 'torchaudio', 'pandas', 'scikit-learn',
    'scikit-image', 'pillow', 'matplotlib', 'transformers', 'tokenizers',
    'huggingface-hub', 'safetensors', 'nibabel', 'SimpleITK', 'tqdm', 'requests',
]

def _installed(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

try:
    _vt_version = metadata.version('voxtell')
except metadata.PackageNotFoundError:
    _vt_version = None

if _vt_version:
    print(f'voxtell {_vt_version} already installed -- skipping pip setup.')
else:
    _constraints = Path('/kaggle/working/.pip_constraints.txt')
    _pins = [f'{p}=={_installed(p)}' for p in _PIN_IF_PRESENT if _installed(p)]
    _constraints.write_text('\n'.join(_pins) + '\n')
    print('Pinned (kept as-is):', ', '.join(_pins))

    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               '-c', str(_constraints), *args])

    # 1) voxtell itself without deps (its torch!=2.9 metadata must not trigger a torch swap)
    _pip('--no-deps', 'voxtell>=0.1.2')
    # 2) only the runtime deps that are actually missing; pinned packages stay untouched
    _pip('nnunetv2>=2.7', 'acvl-utils>=0.2.3,<0.3', 'dynamic-network-architectures>=0.4.1,<0.5',
         'positional-encodings', 'einops', 'SimpleITK', 'nibabel', 'tqdm', 'huggingface_hub')

    importlib.invalidate_caches()
    print('voxtell installed:', metadata.version('voxtell'), '-- no restart needed.')


## 4. Clone the research repo

Uses a Kaggle Secret named `GITHUB_PAT` (Add-ons -> Secrets) -- never hardcode a token here
(an earlier sibling notebook did, and it had to be flagged and rotated).

In [ ]:
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'teamwork/totalsegmentatorV2', '--depth', '1', _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    raise RuntimeError(f'git clone failed (exit {_result.returncode}):\n{_stderr_scrubbed}')

RESEARCH_REPO = Path(_dest)
print('cloned into', RESEARCH_REPO)


## 4b. Download the 50-subject TotalSegmentatorV2 dataset from Hugging Face

Public dataset repo (`Fatema142/totalsegmentator-50-rq-analysis`) -- no Kaggle Secret / token
needed to download it.


In [ ]:
from huggingface_hub import snapshot_download

TOTALSEG_DIR = Path(snapshot_download(
    repo_id='Fatema142/totalsegmentator-50-rq-analysis',
    repo_type='dataset',
    local_dir=str(WORKING_DIR / 'totalsegmentator_50'),
))
print('downloaded to', TOTALSEG_DIR)


## 5. Download and load the VoxTell model

In [ ]:
import torch
from huggingface_hub import snapshot_download
from voxtell.inference.predictor import VoxTellPredictor

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
CKPT_CACHE_DIR = WORKING_DIR / 'checkpoints'
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = str(Path(_local_repo_dir) / VOXTELL_MODEL_VERSION)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
import voxtell.inference.predictor as _vp

class _NoQwen:  # VoxTellPredictor.__init__ would load a full fp32 Qwen3-4B (~16 GB) into host RAM; cell 14
    @staticmethod  # then loads the fp16 one we actually use. Holding both at once can kill the Kaggle kernel.
    def from_pretrained(*a, **k):
        return torch.nn.Identity()

_orig_automodel = _vp.AutoModel
_vp.AutoModel = _NoQwen
try:
    predictor = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
finally:
    _vp.AutoModel = _orig_automodel
# VoxTellPredictor.__init__ loads the checkpoint on CPU and only moves the network to
# `device` inside predict_sliding_window_return_logits() -- every mechanistic helper below
# calls model.encoder/model.decoder directly (bypassing that method), so the network must be
# moved here or every such call hits 'Input type ... and weight type ... should be the same'.
predictor.network = predictor.network.to(device)
model = predictor.network
print('VoxTell predictor ready on', device, '| num_heads (RQ1 fusion channels):', model.num_heads)
assert model.num_heads == 32, f'RQ1 assumes 32 fusion channels, checkpoint has num_heads={model.num_heads}'


## 5b. Fix: text encoder OOM on a single T4/P100 (encoder on CPU + embedding cache)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

if not hasattr(predictor, '_orig_embed_text_prompts'):
    predictor._orig_embed_text_prompts = predictor.embed_text_prompts

# The 4B-param fp16 text encoder (~8 GB) is kept on the CPU, NOT on the GPU: on a 15 GB T4 it left too
# little room for the segmentation network (RQ3's run_interventions died with "out of GPU memory even
# with one output per pass"). Embeddings are cached per prompt string, so each distinct prompt is
# computed once (briefly moving the encoder to the GPU) and every later call is a dictionary lookup --
# no 8 GB CPU<->GPU transfer per call like the original VoxTellPredictor.embed_text_prompts.
predictor.text_backbone = AutoModel.from_pretrained(
    'Qwen/Qwen3-Embedding-4B', dtype=torch.float16  # VoxTell v0.1.0's default text encoder
).eval()
if getattr(predictor, 'tokenizer', None) is None:
    predictor.tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B')

import threading
from voxtell.utils.text_embedding import last_token_pool, wrap_with_instruction

_EMBED_CACHE = {}  # prompt string -> (D,) float32 CPU tensor
_embed_cache_lock = threading.Lock()


@torch.inference_mode()
def _embed_missing(prompts):
    """Embed `prompts` (list of str) with the encoder moved onto the GPU just for this call."""
    dev = predictor.device
    torch.cuda.empty_cache()
    predictor.text_backbone.to(dev)
    try:
        wrapped = wrap_with_instruction(list(prompts))
        tok = predictor.tokenizer(wrapped, padding=True, truncation=True,
                                  max_length=predictor.max_text_length, return_tensors='pt')
        tok = {k: v.to(dev) for k, v in tok.items()}
        out = predictor.text_backbone(**tok)
        emb = last_token_pool(out.last_hidden_state, tok['attention_mask']).float().cpu()
    finally:
        predictor.text_backbone.to('cpu')
        torch.cuda.empty_cache()
    return emb


def _embed_text_prompts_fp16_safe(text_prompts):
    """Same output as VoxTellPredictor.embed_text_prompts: (1, n_prompts, D) float32 on predictor.device."""
    if isinstance(text_prompts, str):
        text_prompts = [text_prompts]
    text_prompts = list(text_prompts)
    with _embed_cache_lock:
        missing = list(dict.fromkeys(t for t in text_prompts if t not in _EMBED_CACHE))
        if missing:
            for t, e in zip(missing, _embed_missing(missing)):
                _EMBED_CACHE[t] = e
        emb = torch.stack([_EMBED_CACHE[t] for t in text_prompts])
    return emb.view(1, len(text_prompts), -1).to(predictor.device)


predictor.embed_text_prompts = _embed_text_prompts_fp16_safe
print('text encoder: fp16 on CPU + per-prompt embedding cache; tokenizer loaded:',
      predictor.tokenizer is not None)


## 6. Helpers (I/O, manifest, RAS axis bookkeeping -- same logic as `scripts/common3d.py`)

In [ ]:
import csv
from pathlib import Path

import numpy as np
from nnunetv2.imageio.nibabel_reader_writer import NibabelIOWithReorient

RESEARCH_REPO = Path(RESEARCH_REPO)
reader = NibabelIOWithReorient()

def load_volume_ras(path):
    """Load a NIfTI image via VoxTell's own reader so predicted masks and ground truth end up
    in the exact same RAS-reoriented array space. Returns (1, X, Y, Z)."""
    data, props = reader.read_images([str(path)])
    return np.asarray(data), props

def spatial_view(arr_4d):
    a = np.asarray(arr_4d)
    while a.ndim > 3 and a.shape[0] == 1:
        a = a[0]
    return a

def load_seg_ras(path):
    data, props = reader.read_seg(str(path))
    return spatial_view(data).astype(np.int32), props

# NibabelIOWithReorient returns axes (Superior-Inferior, Anterior-Posterior, Right-Left) --
# verified empirically against real predictions (left/right kidney centroids differ ~85 voxels
# on this axis, ~3 on the others, for every subject).
AXCODES = ("S", "A", "R")
RL_AXIS = AXCODES.index("R")

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def iou_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return 1.0 if union == 0 else float(inter / union)

print('helpers ready')


## 7. Load the experiment code

Imports `scripts/voxtell_mechanistic.py` (exports `encode_mask_rle`/`decode_mask_rle`/
`encode_array_b64`/`decode_array_b64`, the shared CSV-persistence helpers used by every RQ
below) and `scripts/voxtell_experiments.py` (`run_interventions(..., return_masks=True)`,
reused directly by RQ3/RQ4) from the cloned repo, via `sys.path`.

In [ ]:
import sys

sys.path.insert(0, str(RESEARCH_REPO / 'scripts'))
import voxtell_mechanistic as vm
import voxtell_experiments as ve

print('loaded voxtell_mechanistic from', vm.__file__)
print('loaded voxtell_experiments from', ve.__file__)
assert hasattr(vm, 'encode_mask_rle'), 'need the CSV-persistence helpers in voxtell_mechanistic.py'
assert 'return_masks' in ve.run_interventions.__code__.co_varnames, 'need run_interventions(return_masks=True)'


## 8. Subjects and organs -- ALL 50 staged subjects x every bilateral organ pair

`SUBJECTS` is discovered from the manifest itself (every distinct `subject_id` staged by
`scripts/stage_totalsegmentator_50.py` into `Fatema142/totalsegmentator-50-rq-analysis` -- all 50
subjects, not a hand-picked subset), rather than hardcoded, so this always matches whatever is
actually in the downloaded dataset. `ORGAN_CORES = None` keeps every bilateral organ pair
present in the manifest -- the 19-pair TotalSegmentatorV2 list in
`scripts/stage_totalsegmentator_50.py::BILATERAL_ORGAN_STEMS` (adrenal gland, autochthon,
clavicle, common carotid artery, brachiocephalic vein, femur, gluteus maximus/medius/minimus,
hip, humerus, iliac artery, iliac vein, iliopsoas, kidney, kidney cyst, lung lower/upper lobe,
scapula, subclavian artery). Unilateral structures (lung middle lobe, ribs, vertebrae, skull,
etc.) have no left/right counterpart under this project's scope and are dropped automatically.

**TotalSegmentatorV2 stores one separate binary mask file per organ/side**
(`segmentations/<organ>_<side>.nii.gz` in the source release) -- `load_subject` below assembles
those into one synthetic multi-label volume (a fresh integer id per organ/side, assigned from
the manifest) so every RQ function in `scripts/voxtell_experiments.py`/`voxtell_mechanistic.py`
can work directly on it (they expect a single multi-label `label_vol` + an `ids` dict).

**Running RQ3 over all 50 subjects in one sitting is a lot of GPU time.** If a session times
out, lower `MAX_SUBJECTS` below to a smaller number first, confirm the full notebook runs end to
end, then remove the cap and let it run the full 50 (e.g. across several Kaggle sessions,
appending to the same CSVs since `append_rows` already skips subjects already on disk).


In [ ]:
DATASET_NAME = 'totalsegmentator_v2'
PERSON = 'faria'
ORGAN_CORES = None  # None = every bilateral organ pair present in the manifest
MAX_SUBJECTS = None  # None = all 50 staged subjects; set to e.g. 10 for a quick dry run


def load_manifest_totalsegmentator_50():
    """Loads the 50-subject TotalSegmentatorV2 manifest from the Hugging Face-downloaded
    dataset dir (TOTALSEG_DIR, set in cell 4b). Each row's `organ` column has NO side prefix
    ('adrenal gland', not 'left adrenal gland') -- side lives in its own `side` column -- and
    there is no shared `label_id`: every organ/side has its OWN mask file (`label_path`), so
    `load_subject` below builds one synthetic multi-label volume per subject."""
    path = TOTALSEG_DIR / 'manifest_totalsegmentator_50.csv'
    with open(path, newline='') as f:
        return list(csv.DictReader(f))


_all_rows = load_manifest_totalsegmentator_50()
for _r in _all_rows:
    _r['organ_full'] = f"{_r['side']} {_r['organ']}"  # 'left adrenal gland' -- the 'side organ' form ve.mirror() expects
_all_subject_ids = sorted({r['subject_id'] for r in _all_rows})
assert len(_all_subject_ids) == 50, f'expected 50 staged subjects, manifest has {len(_all_subject_ids)}'
SUBJECTS = [(PERSON, sid) for sid in (_all_subject_ids if MAX_SUBJECTS is None else _all_subject_ids[:MAX_SUBJECTS])]
print(f'using {len(SUBJECTS)} of {len(_all_subject_ids)} staged subjects')

subject_organs = {}
for person, sid in SUBJECTS:
    rows = [r for r in _all_rows if r['subject_id'] == sid]
    if ORGAN_CORES:
        rows = [r for r in rows if r['organ'] in ORGAN_CORES]
    have = {r['organ_full'] for r in rows}
    rows = [r for r in rows if ve.mirror(r['organ_full']) in have]  # keep complete left/right pairs only
    assert rows, f'no bilateral organs for {person}/{sid}'
    # TotalSegmentator has no shared label_id (one mask file per organ/side) -- assign one here,
    # deterministically (sorted by organ_full), so label_vol can be built in load_subject below
    for _i, r in enumerate(sorted(rows, key=lambda r: r['organ_full']), start=1):
        r['label_id'] = _i
    subject_organs[sid] = {'person': person, 'rows': rows}

n_pairs = {sid: len({r['organ'] for r in v['rows']}) for sid, v in subject_organs.items()}
print(len(subject_organs), 'subjects; bilateral organ pairs per subject (first 5 shown):',
      dict(list(n_pairs.items())[:5]))


def load_subject(sid):
    """Returns (person, img_arr, label_vol, organs, ids, prompts) -- the interface every RQ
    function in scripts/voxtell_experiments.py and voxtell_mechanistic.py expects. label_vol is
    assembled here from TotalSegmentator's separate per-organ/side binary mask files (one
    nib.load per organ/side, ~20-40 per subject) into one synthetic multi-label volume."""
    info = subject_organs[sid]
    rows = info['rows']
    base = TOTALSEG_DIR
    img_arr, _ = load_volume_ras(base / rows[0]['image_path'])
    label_vol = None
    for r in rows:
        seg, _ = load_seg_ras(base / r['label_path'])
        if label_vol is None:
            label_vol = np.zeros(seg.shape, dtype=np.int32)
        label_vol[seg > 0] = r['label_id']  # last-writer-wins on overlap (none expected between distinct organs)
    organs = [r['organ_full'] for r in rows]
    ids = {r['organ_full']: r['label_id'] for r in rows}
    prompts = {r['organ_full']: r['prompt_text'] for r in rows}
    return info['person'], img_arr, label_vol, organs, ids, prompts


## 8b. Resumable, crash-safe CSV writer + dual-GPU dispatch

Every RQ loop below writes each subject's rows to CSV **immediately** (not once at the end), and skips subjects already present in its output CSV on startup -- so a Kaggle session timeout/restart/crash loses at most the subject(s) in flight, and rerunning a cell picks up where it left off instead of starting over. If 2 GPUs are visible (Kaggle accelerator "GPU T4 x2"), subjects are split across both and processed concurrently via a second full predictor+model loaded onto the second device.

In [ ]:
import csv
import os
import queue
import threading
import time
import traceback
from pathlib import Path

# Use every visible GPU, not just predictor.device -- set the Kaggle accelerator to "GPU T4 x2"
# (Settings -> Accelerator) to actually get a second device here; with a single-GPU accelerator
# this safely falls back to running everything on predictor.device as before.
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICES = [torch.device(f'cuda:{i}') for i in range(N_GPUS)] if N_GPUS else [predictor.device]
# Safe default: ONE GPU. Building a second VoxTell predictor on cuda:1 is the heaviest step of this cell
# (host RAM + GPU memory) and is what has been killing the Kaggle kernel. Set True only once everything
# runs cleanly on one GPU and you want the extra speed.
USE_SECOND_GPU = False
if not USE_SECOND_GPU:
    DEVICES = [predictor.device]
print('visible CUDA devices:', DEVICES)

PREDICTORS_BY_DEVICE = {predictor.device: predictor}
MODELS_BY_DEVICE = {predictor.device: model}


_embed_lock = threading.Lock()


def _build_predictor_on(device):
    """A second, fully independent VoxTellPredictor (segmentation network + its own
    Qwen3-Embedding-4B text backbone) on `device`, so the RQ loops below can process different
    subjects on different GPUs at the same time instead of everything funneling through the one
    predictor on cuda:0. Uses the same fp16 text backbone and on-device-resident embed_text_prompts
    as cell 14, just bound to this device instead."""
    from voxtell.inference.predictor import VoxTellPredictor
    import voxtell.inference.predictor as _vp

    class _NoQwen:  # the constructor would otherwise load a full fp32 Qwen3-4B (~16 GB) into host RAM,
        @staticmethod  # which can kill the Kaggle kernel; we never use that copy anyway (see below)
        def from_pretrained(*a, **k):
            return torch.nn.Identity()

    _orig_automodel = _vp.AutoModel
    _vp.AutoModel = _NoQwen
    try:
        p = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
    finally:
        _vp.AutoModel = _orig_automodel
    p.network = p.network.to(device)
    # NO second Qwen3 copy on this GPU: a 4B fp16 text encoder is ~8 GB, and on a 15 GB T4 that
    # left no room for the segmentation network's activations (CUDA OOM on every subject). The text
    # encoder is cheap, so all devices share the ONE on the main predictor (serialized by a lock)
    # and the resulting embedding is moved onto this device. The CPU copy VoxTellPredictor's
    # constructor loads is dropped too (frees ~16 GB of host RAM).
    p.text_backbone = None
    main = PREDICTORS_BY_DEVICE[predictor.device]

    def _embed(text_prompts):
        with _embed_lock:
            return main.embed_text_prompts(text_prompts).to(device)

    p.embed_text_prompts = _embed
    return p


import gc
gc.collect()  # drop tensors/graphs still referenced by earlier failed runs (e.g. via stored tracebacks)
for _device in DEVICES:
    if _device.type == 'cuda':
        with torch.cuda.device(_device):
            torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info(_device)
        print(f'{_device}: {_free / 2**30:.1f} / {_total / 2**30:.1f} GiB free before loading')
        if _device not in PREDICTORS_BY_DEVICE and _free < 8 * 2**30:
            raise RuntimeError(f'{_device} only has {_free / 2**30:.1f} GiB free -- something from an earlier '
                               'run is still holding GPU memory. Restart the kernel (Run -> Restart) and re-run '
                               'the setup cells; re-running this cell cannot free it.')

for _device in DEVICES:
    if _device not in PREDICTORS_BY_DEVICE:
        print('loading a second VoxTell + Qwen3 text encoder on', _device, '...')
        PREDICTORS_BY_DEVICE[_device] = _build_predictor_on(_device)
    MODELS_BY_DEVICE[_device] = PREDICTORS_BY_DEVICE[_device].network

WORKER_DEVICES = list(PREDICTORS_BY_DEVICE.keys())
print(f'ready: {len(WORKER_DEVICES)} worker device(s) ->', WORKER_DEVICES)

_csv_lock = threading.Lock()


def completed_ids(csv_path, id_field='subject_id'):
    """Subject IDs already present in csv_path -- lets a rerun after a crash, timeout, or kernel
    restart skip everything that's already on disk instead of starting the whole RQ over."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return set()
    with open(csv_path, newline='') as f:
        return {row[id_field] for row in csv.DictReader(f)}


def append_rows(csv_path, fieldnames, rows):
    """Appends rows to csv_path immediately and fsyncs to disk, writing the header only the first
    time. Called once per subject (not once per whole RQ), so a crash mid-run loses at most the
    subject(s) currently in flight, never the entire run's results."""
    if not rows:
        return
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with _csv_lock:
        write_header = not csv_path.exists()
        with open(csv_path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            if write_header:
                w.writeheader()
            w.writerows(rows)
            f.flush()
            os.fsync(f.fileno())


def run_over_subjects(sids, per_subject_fn, desc=''):
    """Runs per_subject_fn(predictor_d, model_d, device, sid) for every sid in sids, with one
    worker thread per entry in WORKER_DEVICES pulling from a shared queue -- different subjects
    run on different GPUs concurrently. per_subject_fn must call append_rows() itself for every
    CSV it produces, as soon as that subject's rows are ready. Always prints this RQ's total wall
    time (and, for an actual run, the time/subject) so you can see where the budget is going."""
    sids = list(sids)
    t_start = time.time()
    if not sids:
        print(f'[{desc}] nothing to do -- every requested subject is already in the CSV(s) (0s)')
        return
    q = queue.Queue()
    for sid in sids:
        q.put(sid)
    remaining = len(sids)
    progress_lock = threading.Lock()

    def worker(device):
        nonlocal remaining
        predictor_d, model_d = PREDICTORS_BY_DEVICE[device], MODELS_BY_DEVICE[device]
        while True:
            try:
                sid = q.get_nowait()
            except queue.Empty:
                return
            t0 = time.time()
            try:
                per_subject_fn(predictor_d, model_d, device, sid)
                with progress_lock:
                    remaining -= 1
                    left = remaining
                print(f'[{desc}][{device}] {sid}: done in {time.time() - t0:.0f}s ({left} left)')
            except Exception:
                print(f'[{desc}][{device}] {sid}: FAILED')
                traceback.print_exc()
            finally:
                q.task_done()

    threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in WORKER_DEVICES]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    elapsed = time.time() - t_start
    print(f'[{desc}] all {len(sids)} requested subjects processed in {elapsed:.0f}s '
          f'({elapsed / len(sids):.1f}s/subject, {len(WORKER_DEVICES)} GPU(s))')


## 9. Subjects for this run -- the first 50

`SUBJECTS_SUBSET` is the first 50 subjects out of every subject staged in the
manifest (loaded above as `subject_organs`).


In [ ]:
N_SUBJECTS = 50  # <-- change this one number to use more/fewer subjects

_all_loaded_sids = list(subject_organs)
assert N_SUBJECTS <= len(_all_loaded_sids), (
    f'requested {N_SUBJECTS} subjects, but only {len(_all_loaded_sids)} are loaded')

SUBJECTS_SUBSET = _all_loaded_sids[:N_SUBJECTS]
print(f'running the first {len(SUBJECTS_SUBSET)} of {len(_all_loaded_sids)} loaded subjects '
      f'-> {SUBJECTS_SUBSET}')


---
# RQ3 -- Does the prompt decoder preserve or suppress laterality information?

**What I'm doing:** for every bilateral organ pair, measuring how different the left-prompt and
right-prompt query states are at **every one of the 7 points** in the transformer decoder's
residual stream (layer 0 = right after `project_text_embed`, i.e. essentially the raw text
embedding projected into query space; layers 1-6 = after each of the 6 cross-attention/FFN
layers). Reuses `scripts/voxtell_experiments.py::run_interventions`'s `row16`
(`LayerDistanceRecorder`), which already computes exactly this (cosine distance + relative L2
distance), for both the organ pair itself and a different-organ, same-side reference/control
pair. Adds a genuine **KL divergence** between the left- and right-prompt cross-attention
distributions at each of the 6 layers (cross-attention weights already sum to 1 over image
tokens, so this is an exact KL, not an approximation) -- the docx's explicit "KL divergence"
ask, which `row16` alone doesn't cover.

**Why:** RQ1 showed whether the *final* fused representation discriminates laterality; RQ3 asks
whether that discrimination was already present at layer 0 and simply passed through, or whether
the 6-layer decoder itself *creates* it, *preserves* it, or *suppresses* it (collapses left/right
toward each other) somewhere along the way. This is the direct mechanistic link between RQ2's
"text alone" measurement and RQ1's "after fusion" measurement.

**What I'll get (CSV columns):**
- `rq3_layer_distance_and_kl.csv`: one row per subject x pair x scope x layer (0-6) --
  `pair_kind` is `'target'` (the organ's own left-vs-right pair) or `'reference'` (the
  different-organ control pair plot #2 needs -- both are included, so the overlay plot has
  something to draw); `cosine_distance`, `relative_l2_distance` (both from `row16`, `scope` is
  `organ_patches` or `all_patches`), and (for `pair_kind='target'`, layers 1-6 only, since KL
  isn't computed for the cross-organ reference pair) `attention_kl_left_from_right` /
  `attention_kl_right_from_left`.

**What to plot:**
1. **Line plot**, x = layer (0-6), y = `cosine_distance`, one line per organ (or the mean across
   organs with a shaded range) -- a line that **falls** toward layer 6 means the decoder
   *suppresses* laterality (preserves the plan's "becomes less distinguishable" case); a line
   that **rises or stays flat** means it *preserves or amplifies* it.
2. **Same line plot for the `pair` = reference/control** (different organ, same side) overlaid
   as a dashed line -- the gap between the real pair's line and the control's line is the actual
   laterality-specific signal, separated from generic organ-to-organ distance.
3. **Line plot** of `attention_kl_left_from_right` (and its mirror) across layers 1-6 -- a
   complementary, strictly-probabilistic view of the same question, since cross-attention
   weights are true distributions over image tokens.

**Metrics:** cosine distance (`1 - cosine_similarity`), relative L2 distance (both already
normalized so they're comparable across layers/organs), and KL divergence (bidirectional, since
KL isn't symmetric) between cross-attention distributions.

In [ ]:
import torch.nn.functional as F

rq3_csv = RESULTS_ROOT / 'rq3' / 'rq3_layer_distance_and_kl.csv'
rq3_fields = ['dataset', 'person', 'subject_id', 'pair', 'pair_kind', 'scope', 'layer', 'n_patches',
              'cosine_distance', 'relative_l2_distance', 'attention_kl_left_from_right',
              'attention_kl_right_from_left']


@torch.no_grad()  # vm.encode has no grad guard; without this each call keeps the encoder's autograd graph (OOM)
def attention_kl_per_layer(model_d, patch_t, emb_left_row, emb_right_row):
    """emb_*_row: (1, D). Returns (kl_l_from_r, kl_r_from_l), each a length-6 list -- one real
    KL divergence per decoder layer between that layer's left-prompt and right-prompt
    cross-attention distribution over image tokens (both already sum to 1 by construction, so
    no normalization/approximation is needed before computing KL)."""
    skips, memory = vm.encode(model_d, patch_t)
    tgt0_l = model_d.project_text_embed(emb_left_row).unsqueeze(1)
    tgt0_r = model_d.project_text_embed(emb_right_row).unsqueeze(1)
    _, attns_l = vm.base_trajectory(model_d, tgt0_l, memory, model_d.pos_embed)
    _, attns_r = vm.base_trajectory(model_d, tgt0_r, memory, model_d.pos_embed)
    kl_lr, kl_rl = [], []
    for a_l, a_r in zip(attns_l, attns_r):
        p = a_l[0, 0].float().clamp_min(1e-12)
        q = a_r[0, 0].float().clamp_min(1e-12)
        kl_lr.append(float((p * (p.log() - q.log())).sum()))
        kl_rl.append(float((q * (q.log() - p.log())).sum()))
    return kl_lr, kl_rl


def _rq3_one_subject(predictor_d, model_d, device, sid):
    rows = []
    person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
    # row16 (cosine_distance / relative_l2_distance per layer) comes straight from
    # run_interventions -- no need to re-derive it here
    # RQ3 only reads res['row16'] (the per-layer distances recorded on the first pass). The blend /
    # single-layer-update / updates-from-layer / whole-state conditions are RQ4's job and made up ~93% of
    # the outputs (~430 of ~460 per subject, each needing its own chunked sliding-window pass), so they
    # are switched off here -- only the baselines run. Same row16 values, far fewer GPU passes.
    res = ve.run_interventions(predictor_d, img_arr, label_vol, organs, ids, prompts, rl_axis=RL_AXIS,
                               alphas=(), single_layers=(), from_layers=(), state_layers=())
    row16_by_key = {(r['pair'], r['scope'], r['layer']): r for r in res['row16']}
    # row16_by_key holds BOTH the target ("X: left vs right") and the different-organ
    # reference/control pair ("X: left X vs left OTHER (reference)") that run_interventions
    # already computed -- found here by name, not recomputed, so there's no risk of this
    # notebook's own reference-pair selection disagreeing with run_interventions's
    all_pair_names = {r['pair'] for r in res['row16']}

    data, bbox, orig_shape = predictor_d.preprocess(img_arr)
    left_organs = sorted(o for o in organs if o.startswith('left') and ve.mirror(o) in organs)
    for organ in left_organs:
        right_organ = ve.mirror(organ)
        core = organ.split(None, 1)[1]
        c_left = ve.centroid((label_vol[tuple(slice(int(b[0]), int(b[1])) for b in bbox)]) == ids[organ])
        c_right = ve.centroid((label_vol[tuple(slice(int(b[0]), int(b[1])) for b in bbox)]) == ids[right_organ])
        if c_left is None or c_right is None:
            continue
        centre = tuple((a + b) / 2 for a, b in zip(c_left, c_right))
        patch, offset = ve._crop_patch_around(data, centre, predictor_d.patch_size)
        patch_t = (patch if torch.is_tensor(patch) else torch.from_numpy(patch)).to(device)
        _emb_both = predictor_d.embed_text_prompts([prompts[organ], prompts[right_organ]])[0]  # (2, D)
        emb_l, emb_r = _emb_both[0:1].to(device), _emb_both[1:2].to(device)
        kl_lr, kl_rl = attention_kl_per_layer(model_d, patch_t, emb_l, emb_r)

        target_name = f'{core}: left vs right'
        reference_name = next((p for p in all_pair_names
                                if p.startswith(f'{core}: ') and p.endswith('(reference)')), None)
        # KL is only computed for the target (left/right) pair above -- the reference pair is
        # two DIFFERENT organs' prompts, not a left/right pair, so there's no natural
        # left-vs-right KL to compute for it; its cosine/L2 distance (from row16) is still
        # saved below so the control-pair overlay plot in the markdown has data to draw.
        for pair_name, pair_kind, kl_for_this_pair in ((target_name, 'target', (kl_lr, kl_rl)),
                                                        (reference_name, 'reference', (None, None))):
            if pair_name is None:
                continue
            for scope in ('organ_patches', 'all_patches'):
                for layer in range(7):
                    r16 = row16_by_key.get((pair_name, scope, layer))
                    kl_lr_this, kl_rl_this = kl_for_this_pair
                    rows.append({
                        'dataset': DATASET_NAME, 'person': person, 'subject_id': sid, 'pair': pair_name,
                        'pair_kind': pair_kind, 'scope': scope, 'layer': layer,
                        'n_patches': r16['n_patches'] if r16 else None,
                        'cosine_distance': r16['cosine_distance'] if r16 else None,
                        'relative_l2_distance': r16['relative_l2_distance'] if r16 else None,
                        'attention_kl_left_from_right': round(kl_lr_this[layer - 1], 6) if kl_lr_this and 1 <= layer <= 6 else None,
                        'attention_kl_right_from_left': round(kl_rl_this[layer - 1], 6) if kl_rl_this and 1 <= layer <= 6 else None,
                    })

    append_rows(rq3_csv, rq3_fields, rows)


_rq3_done = completed_ids(rq3_csv)
_rq3_todo = [sid for sid in SUBJECTS_SUBSET if sid not in _rq3_done]
print(f'RQ3: {len(_rq3_done)} subjects already in {rq3_csv.name}, {len(_rq3_todo)} left to run')
run_over_subjects(_rq3_todo, _rq3_one_subject, desc='rq3')


---
## Outputs -- RQ3 CSVs for 50 subjects

Masks are in the **full volume** coordinate frame (`volume_shape` column, varies per
subject) -- decode with `vm.decode_mask_rle(cell, volume_shape_tuple)`, where
`volume_shape_tuple = tuple(int(x) for x in row['volume_shape'].split('x'))`.

| RQ | File | Columns worth knowing | Decode masks with |
|---|---|---|---|
| RQ3 | `rq3_layer_distance_and_kl.csv` | `cosine_distance`/`relative_l2_distance`/`attention_kl_*` per layer 0-6 | -- |

Rows are keyed by `subject_id`, so results from separate runs (e.g. different
`MAX_SUBJECTS` caps, or resumed after a Kaggle session timeout) can be combined with a plain
`pandas.concat` + `drop_duplicates(subset=['subject_id', ...])`.
